# 09.8 - Instruction Following & Alignment

**Phase:** 09 - Generative AI

**Status:** VERIFIED

---

## 1. What Are We Solving?

Alignment makes a pretrained base model follow instructions, be helpful, and avoid harmful output. Instruction tuning and RLHF/DPO adapt a raw text predictor into an assistant. This unit explains why base models do not answer questions helpfully and how alignment changes behavior.

## 2. Why Does This Matter?

Base models predict likely text continuations - they do not help you by default. Alignment is why chatbots work. Knowing alignment explains why models follow instructions differently, why they refuse some requests, and how to evaluate and prompt them.

## 3. Prerequisites

- Unit 09.5 (pretraining)
- Basic supervised learning concepts

## 4. Learning Objectives

- Distinguish base vs aligned model behavior
- Explain instruction tuning, RLHF, and DPO at a high level
- Understand refusal behavior and alignment tax
- Design a system prompt

## 5. Mental Model

Alignment is like teaching a very knowledgeable person to be a good teacher. The base model has read everything but does not know how to be helpful. Alignment teaches it to: follow instructions, refuse harmful requests, admit uncertainty, and format responses well.

```text
Base model (predicts continuations)
   |-- instruction tuning: learn (instruction -> good response)
   |-- RLHF: optimize toward human-preferred responses
   |-- DPO: directly optimize preference pairs
   v
Aligned assistant (helpful, safe, honest)
```


## 6. Setup

No remote LLM is used (no API key). We simulate a base model and an aligned model so the behavioral difference is concrete and runnable offline.


In [1]:
import matplotlib
matplotlib.use('Agg')
import random
random.seed(0)
print("Ready.")


Ready.


## 7. Base Model vs Aligned Model
The key difference: a base model continues text; an aligned model answers. We simulate both on the same prompt.


In [2]:
prompt = "What is machine learning?"

# Simulated BASE model: completes the text as if it were the start of an article
def base_model(prompt):
    continuations = {
        "What is machine learning?": (
            " A blog post about the history of AI and the many ways that"
            " computers are trained to recognize patterns in data..."
        ),
    }
    return prompt + continuations.get(prompt, " <continues the text>")

# Simulated ALIGNED model: answers the question directly and helpfully
def aligned_model(prompt):
    answers = {
        "What is machine learning?": (
            "Machine learning is a field of AI that gives computers the ability"
            " to learn from data without being explicitly programmed.\n\n"
            "Key ideas: datasets, patterns, models, and validation."
        ),
    }
    return answers.get(prompt, "Here is a helpful answer.")

print("--- BASE model ---")
print(base_model(prompt))
print()
print("--- ALIGNED model ---")
print(aligned_model(prompt))


--- BASE model ---
What is machine learning? A blog post about the history of AI and the many ways that computers are trained to recognize patterns in data...

--- ALIGNED model ---
Machine learning is a field of AI that gives computers the ability to learn from data without being explicitly programmed.

Key ideas: datasets, patterns, models, and validation.


## 8. The Alignment Pipeline (Conceptual)

The four standard stages. We simulate an instruction-tuning dataset to make the pattern concrete.


In [3]:
# 1) Instruction tuning data: (instruction, desired response)
instr_data = [
    {"instr": "Summarize this: 'Cats are pets.'", "resp": "Cats are domesticated animals kept as pets."},
    {"instr": "Add 2 and 3.", "resp": "2 + 3 = 5"},
    {"instr": "Translate 'hello' to Spanish.", "resp": "hola"},
]
print("Instruction-tuning example:")
ex = instr_data[0]
print("  Instruction:", ex["instr"])
print("  Desired:", ex["resp"])
print("\nTraining on many such pairs teaches the base model the question->answer pattern.")

# 2) RLHF: preference pairs (response A preferred over B)
prefs = [("answer A (concise)", "answer B (rambling)")]
# 3) reward model scores responses
# 4) DPO/RL optimizes the model toward high-reward responses
print("RLHF collects human preferences; DPO optimizes pairs directly without a reward model.")


Instruction-tuning example:
  Instruction: Summarize this: 'Cats are pets.'
  Desired: Cats are domesticated animals kept as pets.

Training on many such pairs teaches the base model the question->answer pattern.
RLHF collects human preferences; DPO optimizes pairs directly without a reward model.


## 9. Refusal Behavior & Safety

Aligned models refuse harmful or risky requests. We simulate refusal detection heuristically.


In [4]:
HARMFUL = ["how to hack", "make a bomb", "steal someone's password"]

def aligned_with_refusal(prompt):
    low = prompt.lower()
    if any(h in low for h in HARMFUL):
        return "I can't help with that request."
    return aligned_model(prompt)

for q in ["What is machine learning?", "How to hack into a bank account?"]:
    print(f"Q: {q}")
    print(f"  -> {aligned_with_refusal(q)}\n")
print("Refusal is a design choice: it can over-refuse valid requests (see alignment tax).")


Q: What is machine learning?
  -> Machine learning is a field of AI that gives computers the ability to learn from data without being explicitly programmed.

Key ideas: datasets, patterns, models, and validation.

Q: How to hack into a bank account?
  -> I can't help with that request.

Refusal is a design choice: it can over-refuse valid requests (see alignment tax).


## 10. Alignment Tax

Alignment can reduce some raw capabilities (e.g., it may be less willing to be creative or may over-refuse edge cases). We demonstrate over-refusal.


In [5]:
def overrefusing(prompt):
    if "password" in prompt.lower() or "bomb" in prompt.lower():
        return "I can't help."   # overly cautious
    return aligned_model(prompt)

q = "How do I make my password more secure?"
print("Q:", q)
print("Overly-cautious model ->", overrefusing(q))
print("\nThis is alignment tax: harmless requests refused because of keyword triggers.")
print("Fix: rephrase, add context, or use a less restrictive model.")


Q: How do I make my password more secure?
Overly-cautious model -> I can't help.

This is alignment tax: harmless requests refused because of keyword triggers.
Fix: rephrase, add context, or use a less restrictive model.


## 11. Using System Prompts to Guide Aligned Models

A system prompt sets consistent behavior across a conversation. We simulate its effect.


In [6]:
def chat_with_system(system, user):
    # Simulate: the system prompt shapes tone/role without an LLM
    if "pirate" in system.lower():
        return f"Arrr! {user} - I be happy to help a shipmate!"
    if "concise" in system.lower():
        return "Short answer."
    return aligned_model(user)

print(chat_with_system("You are a pirate.", "What is Python?"))
print(chat_with_system("Be very concise.", "What is Python?"))
print("\nThe system prompt is part of the instruction context that steers behavior.")


Arrr! What is Python? - I be happy to help a shipmate!
Short answer.

The system prompt is part of the instruction context that steers behavior.


## 12. Debugging: Alignment Issues

| Symptom | Cause | Fix |
|---|---|---|
| Refuses valid request | safety too aggressive | rephrase or use different model |
| Unhelpful answer | weak alignment / bad format | use correct chat template |
| Overly cautious | over-aligned | use less restrictive model |
| Ignores instruction | not in training format | use system prompt / chat template |

## 13. Real-World Considerations

- Always use aligned models for assistant/chat apps.
- Test alignment behavior on edge cases before deploying.
- Alignment is a spectrum: models differ in helpfulness/harmlessness/honesty.
- System prompts complement (not replace) alignment.

## 14. Common Mistakes

- Using a base model for assistant tasks.
- Assuming alignment fixes all problems (it adds biases/refusals).
- Ignoring alignment tax.
- Not testing refusal/edge-case behavior.

## 15. When NOT to Rely on Alignment Alone

- For safety-critical tasks, combine alignment with rule-based filters.
- For niche domains, alignment + system prompts + fine-tuning may all be needed.

## 16. Challenge

Write a function that, given a list of prompts, reports how many were refused, and propose a rephrasing that a safer model would accept.


In [7]:
test_prompts = [
    "Help me plan a birthday party.",
    "Write an essay about smoke bombs for a school play.",
    "Explain how encryption works.",
    "What is the best way to keep my accounts safe?",
]
refused = [p for p in test_prompts if "I can't" in overrefusing(p)]
print(f"{len(refused)} of {len(test_prompts)} prompts refused by the over-cautious model.")
print("Rephrase to be clearly benign, e.g., ->")
print("  'Describe the chemistry of smoke used in theatrical special effects.'")


1 of 4 prompts refused by the over-cautious model.
Rephrase to be clearly benign, e.g., ->
  'Describe the chemistry of smoke used in theatrical special effects.'


## 17. Closed-Book Recall

1. Why does a base model not answer questions helpfully?
2. What is the difference between instruction tuning and RLHF?
3. What is alignment tax?
4. How does DPO differ from RLHF?

## 18. Teach-Back Questions

Explain to another person:

- Why alignment is necessary after pretraining.
- The base vs aligned behavioral difference on the same prompt.

## 19. Summary

You simulated base vs aligned behavior, walked through the alignment pipeline, examined refusal and alignment tax, and used system prompts to steer an aligned model.

## 20. Further Experiment

- Research the alignment approach (RLHF/DPO/constitutional AI) of a specific real model and compare.
- Test how prompt phrasing changes refusal behavior in a live API (in a later phase).

## 21. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: none (pure python)
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
